In [5]:
# --- llm_processor.py logic: key point extraction ---
import os
from dataclasses import dataclass
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

@dataclass
class KeyPoints:
    summary: str
    source_title: str


SYSTEM_PROMPT = """You are a research analyst who prepares source material for a podcast writer.
Given the full text of an academic paper, extract the key points a general/educated audience
would find interesting. make the tone interesting engaging. 
Focus on: the core problem/motivation, main findings or contributions,
important methods or approaches (explained simply, no heavy jargon), real-world implications,
and open challenges or future directions.
Ignore: author affiliations, funding acknowledgments, and any reference/citation lists.
Output as clear, organized bullet points grouped under short headers
(e.g., "Motivation", "Key Findings", "Implications", "Open Challenges").
Keep it concise, aim for 400-700 words total."""


def extract_key_points(text: str, title: str = "", model: str = "gpt-4o-mini") -> KeyPoints:
    if not text or len(text.strip()) < 200:
        raise ValueError("Input text is too short to extract meaningful key points.")

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Title: {title}\n\nPaper text:\n{text}"},
            ],
            temperature=0.3,
        )
    except Exception as e:
        raise RuntimeError(f"LLM API call failed: {e}")

    summary = response.choices[0].message.content.strip()

    if not summary:
        raise RuntimeError("LLM returned an empty response.")

    return KeyPoints(summary=summary, source_title=title)

In [6]:

# --- test ---
if __name__ == "__main__":
    with open("output/full_text.txt", "r", encoding="utf-8") as f:
        clean_text = f.read()

    result = extract_key_points(
        clean_text,
        title="Non-Invasive Brain-Computer Interfaces: State of the Art and Trends"
    )
    print(result.summary)

    with open("output/key_points.txt", "w", encoding="utf-8") as f:
        f.write(result.summary)

### Motivation
- **What are BCIs?** Brain-computer interfaces (BCIs) are technologies that allow individuals to communicate directly with external devices using brain activity, bypassing traditional pathways like muscle movement.
- **Why Non-Invasive?** Non-invasive BCIs are particularly appealing due to their safety, affordability, and ability to engage a broader population, including those with neuromuscular impairments like ALS or spinal cord injuries.
- **Historical Context:** Initially, BCIs were limited to simple tasks, but advancements now enable control of complex devices, such as robotic arms, enhancing daily life for users.

### Key Findings
- **Current Applications:** Non-invasive BCIs are increasingly used not just in clinical settings but also in recreational contexts, such as brain games, expanding their reach and acceptance.
- **Technological Advancements:** Modern BCIs can decode complex brain signals, allowing for direct control of devices. Innovations include handwrit

In [7]:
# --- llm_processor.py logic: script generation ---

from dataclasses import dataclass

SCRIPT_SYSTEM_PROMPT = """You are a podcast scriptwriter. Turn the key points below into a natural,
engaging, single-narrator podcast script for a general/educated audience interested in science
and technology.Stress the potential of the technology.

Requirements:
- Write in a conversational, spoken tone (contractions, natural rhythm); NOT an essay or bullet list.
- Structure: a short welcoming intro (hook the listener, state what the episode covers),
  a well-paced main body walking through the key points with natural transitions between ideas,
  and a brief outro that wraps up with a takeaway and sign-off.
- Do not use markdown formatting, headers, or bullet points; this text will be read aloud by
  a text-to-speech engine, so it must be plain spoken prose only.
- Avoid citation-style references (no "[1]", "et al.", etc.); just explain ideas in plain language.
- Target length: roughly 600-900 words (up to 5 minutes of spoken audio)."""


@dataclass
class PodcastScript:
    script: str
    title: str


def generate_script(key_points: str, title: str = "", model: str = "gpt-4o-mini") -> PodcastScript:
    if not key_points or len(key_points.strip()) < 50:
        raise ValueError("Key points input is too short to generate a script from.")

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": SCRIPT_SYSTEM_PROMPT},
                {"role": "user", "content": f"Topic: {title}\n\nKey points:\n{key_points}"},
            ],
            temperature=0.7,  # a bit more creative/natural for spoken tone
        )
    except Exception as e:
        raise RuntimeError(f"LLM API call failed: {e}")

    script_text = response.choices[0].message.content.strip()

    if not script_text:
        raise RuntimeError("LLM returned an empty script.")

    return PodcastScript(script=script_text, title=title)


In [8]:
# --- quick test ---
if __name__ == "__main__":
    with open("output/key_points.txt", "r", encoding="utf-8") as f:
        key_points = f.read()

    result = generate_script(
        key_points,
        title="Non-Invasive Brain-Computer Interfaces: State of the Art and Trends"
    )
    print(result.script)

    with open("output/podcast_script.txt", "w", encoding="utf-8") as f:
        f.write(result.script)

Welcome to today’s episode, where we’re diving into the fascinating world of non-invasive brain-computer interfaces, or BCIs for short. Now, I know that sounds like something straight out of a sci-fi movie, but trust me, it’s very much a reality today. So, what are these BCIs exactly? Well, they’re technologies that allow us to communicate directly with external devices using our brain activity, bypassing the usual routes like muscle movement. Imagine that! 

What makes non-invasive BCIs particularly attractive is their safety and affordability. Unlike their invasive counterparts, which require surgical procedures, non-invasive BCIs can engage a broader range of people, especially those with neuromuscular impairments, like ALS or spinal cord injuries. It’s a game changer in accessibility. 

Historically, BCIs were limited to performing simple tasks, but advancements over the years have opened up a world of possibilities. Now, users can control complex devices, such as robotic arms, whi